# Imports

In [1]:
import numpy as np
import matplotlib
from matplotlib import rc
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import scipy
from scipy.ndimage import shift
from scipy.ndimage import gaussian_filter
import scipy.io as sio
from scipy.io import readsav
import csv

#below are ones that I added
import pandas as pd
from bokeh.io import push_notebook, show, output_notebook
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, Label, LabelSet,BoxAnnotation
output_notebook()


#Global Variables (This is bad, do not emulate my bad programming)
NORMALIZATION_INDEX = 800 #420 #I dunno pick one

Loading BokehJS ...

## Data Load-In

In [2]:
N1_R1_A = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_19sep2025/venus_1118_a_ortho_19sep25.sav')
N1_R1_B = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_19sep2025/venus_1118_b_ortho_19sep25.sav')
N1_R1_C = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_19sep2025/venus_1118_c_ortho_19sep25.sav')
N1_R1_tot = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_19sep2025/venus_1118_total_ortho_19sep25.sav')

N2_R1_A = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_21sep2025/venus_1118_a_ortho_21sep25.sav')
N2_R1_B = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_21sep2025/venus_1118_b_ortho_21sep25.sav')
N2_R1_C = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_21sep2025/venus_1118_c_ortho_21sep25.sav')
N2_R1_tot = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_21sep2025/venus_1118_total_ortho_21sep25.sav')

In [3]:
N1_R2_A = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_19sep2025/venus_1207_a_ortho_19sep25.sav')
N1_R2_B = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_19sep2025/venus_1207_b_ortho_19sep25.sav')
N1_R2_tot = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_19sep2025/venus_1207_total_ortho_19sep25.sav')

N2_R2_A = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_21sep2025/venus_1207_a_ortho_21sep25.sav')
N2_R2_B = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_21sep2025/venus_1207_b_ortho_21sep25.sav')
N2_R2_C = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_21sep2025/venus_1207_c_ortho_21sep25.sav')
N2_R2_tot = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_21sep2025/venus_1207_total_ortho_21sep25.sav')

In [4]:
#Figure out what we got
print(N1_R1_tot.keys())
print(N2_R1_tot.mapgrid)

dict_keys(['spec_cube', 'sky', 'wavenumber', 'hdr', 'latitude', 'longitude', 'airmass', 'vel_map', 'mapgrid', 'inttime', 'map_pixel_scale', 'contmap', 'emismap'])
[-20.5 -20.  -19.5 -19.  -18.5 -18.  -17.5 -17.  -16.5 -16.  -15.5 -15.
 -14.5 -14.  -13.5 -13.  -12.5 -12.  -11.5 -11.  -10.5 -10.   -9.5  -9.
  -8.5  -8.   -7.5  -7.   -6.5  -6.   -5.5  -5.   -4.5  -4.   -3.5  -3.
  -2.5  -2.   -1.5  -1.   -0.5   0.    0.5   1.    1.5   2.    2.5   3.
   3.5   4.    4.5   5.    5.5   6.    6.5   7.    7.5   8.    8.5   9.
   9.5  10.   10.5  11.   11.5  12.   12.5  13.   13.5  14.   14.5  15.
  15.5  16.   16.5  17.   17.5  18.   18.5  19.   19.5  20.   20.5]


In [5]:
print(N1_R2_tot.keys())
print(N2_R2_tot.mapgrid)

dict_keys(['spec_cube', 'sky', 'wavenumber', 'hdr', 'latitude', 'longitude', 'airmass', 'vel_map', 'mapgrid', 'inttime', 'map_pixel_scale', 'contmap', 'emismap'])
[-20.5 -20.  -19.5 -19.  -18.5 -18.  -17.5 -17.  -16.5 -16.  -15.5 -15.
 -14.5 -14.  -13.5 -13.  -12.5 -12.  -11.5 -11.  -10.5 -10.   -9.5  -9.
  -8.5  -8.   -7.5  -7.   -6.5  -6.   -5.5  -5.   -4.5  -4.   -3.5  -3.
  -2.5  -2.   -1.5  -1.   -0.5   0.    0.5   1.    1.5   2.    2.5   3.
   3.5   4.    4.5   5.    5.5   6.    6.5   7.    7.5   8.    8.5   9.
   9.5  10.   10.5  11.   11.5  12.   12.5  13.   13.5  14.   14.5  15.
  15.5  16.   16.5  17.   17.5  18.   18.5  19.   19.5  20.   20.5]


In [6]:
#Print the wavenumber ranges
print("Region 1 Night 1:")
print(str(N1_R1_tot['wavenumber'][0]) + ' - ' + str(N1_R1_tot['wavenumber'][-1]))
print("Region 1 Night 2:")
print(str(N2_R1_tot['wavenumber'][0]) + ' - ' + str(N2_R1_tot['wavenumber'][-1]))

print("Region 2 Night 1:")
print(str(N1_R2_tot['wavenumber'][0]) + ' - ' + str(N1_R2_tot['wavenumber'][-1]))
print("Region 2 Night 2:")
print(str(N2_R2_tot['wavenumber'][0]) + ' - ' + str(N2_R2_tot['wavenumber'][-1]))

Region 1 Night 1:
1113.9566362863977 - 1122.815079266173
Region 1 Night 2:
1114.6205838457513 - 1122.8187838770396
Region 2 Night 1:
1204.0046390186249 - 1210.2899295889044
Region 2 Night 2:
1203.3513422910116 - 1210.2995432767207


## Line-by-Line Load In

In [213]:
#import line by line (assuming gathered from HITRAN)
df=pd.read_fwf('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/line_by_lines/H2O_all.txt',header=None) 
wn=df.loc[:,1]
abund=df.loc[:,2]
split_df=pd.concat([wn,abund],axis=1)
split_array=split_df.to_numpy()
#print(split_array)

#get desired wavenumbers

mol_lambda1=[]
abund_lambda1=[]
for a in range(0,len(split_array),1):
    if split_array[a,0] > 1114.7114576040406 and split_array[a,0] < 1123.696475590839:
    #if split_array[a,0] > 1121 and split_array[a,0] < 1123:
        if split_array[a,1] > 1e-23: # and split_array[a,1] < 1e-22: #remove the and for the regular limit
            mol_lambda1.append(split_array[a,0]) #get list of wavenumber locations
            abund_lambda1.append(split_array[a,1]) # get corresponding strengths

print(len(mol_lambda1))
range1_array=np.stack((mol_lambda1, abund_lambda1),axis=-1) #make locations and strengths one array
line_locations_region_1 = np.array(mol_lambda1) #make it match previous naming structure

2


In [216]:
#import line by line (assuming gathered from HITRAN)
df=pd.read_fwf('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/line_by_lines/O3_all.txt',header=None) 
wn=df.loc[:,1]
abund=df.loc[:,2]
split_df=pd.concat([wn,abund],axis=1)
split_array=split_df.to_numpy()
#print(split_array)

#get desired wavenumbers

#Region 1:
two_mol_lambda1=[]
two_abund_lambda1=[]
for a in range(0,len(split_array),1):
    if split_array[a,0] > 1114.7114576040406 and split_array[a,0] < 1123.696475590839:
    #if split_array[a,0] > 1117 and split_array[a,0] < 1118 or split_array[a,0] > 1119 and split_array[a,0] < 1119.33:
        if split_array[a,1] > 1e-22: # and split_array[a,1] < 1e-22: #remove the and for the regular limit
            two_mol_lambda1.append(split_array[a,0]) #get list of wavenumber locations
            two_abund_lambda1.append(split_array[a,1]) # get corresponding strengths

print(len(two_mol_lambda1))
two_range1_array=np.stack((two_mol_lambda1, two_abund_lambda1),axis=-1) #make locations and strengths one array
two_line_locations_region_1 = np.array(two_mol_lambda1) #make it match previous naming structure

43


# Global Spectra

## Variables

### Region One

In [7]:
disk_N1_R1_A =np.argwhere(N1_R1_A.airmass>0)
disk_N1_R1_B =np.argwhere(N1_R1_B.airmass>0)
disk_N1_R1_C =np.argwhere(N1_R1_C.airmass>0)
disk_N1_R1_tot =np.argwhere(N1_R1_tot.airmass>0)

disk_N2_R1_A =np.argwhere(N2_R1_A.airmass>0)
disk_N2_R1_B =np.argwhere(N2_R1_B.airmass>0)
disk_N2_R1_C =np.argwhere(N2_R1_C.airmass>0)
disk_N2_R1_tot =np.argwhere(N2_R1_tot.airmass>0)

In [8]:
vlambda_N1_R1_A = []
vflux_N1_R1_A = []
for i in range(len(N1_R1_A['wavenumber'])):
    vlambda_N1_R1_A.append(N1_R1_A['wavenumber'][i])
    sum_N1_R1_A=0
    for j in range(len(disk_N1_R1_A)): #disk[j,0]:
        row=disk_N1_R1_A[j,0]
        col=disk_N1_R1_A[j,1]
        sum_N1_R1_A = sum_N1_R1_A + N1_R1_A['spec_cube'][int(row),int(col),int(i)]
    vflux_N1_R1_A.append(sum_N1_R1_A)

slambda_N1_R1_A = []
sflux_N1_R1_A = []

for i in range(len(N1_R1_A['wavenumber'])):
    slambda_N1_R1_A.append(N1_R1_A['wavenumber'][i])
    sum_N1_R1_A=0
    for j in range(len(disk_N1_R1_A)): #disk[j,0]:
        row=disk_N1_R1_A[j,0]
        col=disk_N1_R1_A[j,1]
        sum_N1_R1_A = sum_N1_R1_A + N1_R1_A['sky'][int(row),int(col),int(i)]
    sflux_N1_R1_A.append(sum_N1_R1_A)

In [9]:
vlambda_N1_R1_B = []
vflux_N1_R1_B = []
for i in range(len(N1_R1_B['wavenumber'])):
    vlambda_N1_R1_B.append(N1_R1_B['wavenumber'][i])
    sum_N1_R1_B=0
    for j in range(len(disk_N1_R1_B)): #disk[j,0]:
        row=disk_N1_R1_B[j,0]
        col=disk_N1_R1_B[j,1]
        sum_N1_R1_B = sum_N1_R1_B + N1_R1_B['spec_cube'][int(row),int(col),int(i)]
    vflux_N1_R1_B.append(sum_N1_R1_B)

slambda_N1_R1_B = []
sflux_N1_R1_B = []

for i in range(len(N1_R1_B['wavenumber'])):
    slambda_N1_R1_B.append(N1_R1_B['wavenumber'][i])
    sum_N1_R1_B=0
    for j in range(len(disk_N1_R1_B)): #disk[j,0]:
        row=disk_N1_R1_B[j,0]
        col=disk_N1_R1_B[j,1]
        sum_N1_R1_B = sum_N1_R1_B + N1_R1_B['sky'][int(row),int(col),int(i)]
    sflux_N1_R1_B.append(sum_N1_R1_B)

In [10]:
vlambda_N1_R1_C = []
vflux_N1_R1_C = []
for i in range(len(N1_R1_C['wavenumber'])):
    vlambda_N1_R1_C.append(N1_R1_C['wavenumber'][i])
    sum_N1_R1_C=0
    for j in range(len(disk_N1_R1_C)): #disk[j,0]:
        row=disk_N1_R1_C[j,0]
        col=disk_N1_R1_C[j,1]
        sum_N1_R1_C = sum_N1_R1_C + N1_R1_C['spec_cube'][int(row),int(col),int(i)]
    vflux_N1_R1_C.append(sum_N1_R1_C)

slambda_N1_R1_C = []
sflux_N1_R1_C = []

for i in range(len(N1_R1_C['wavenumber'])):
    slambda_N1_R1_C.append(N1_R1_A['wavenumber'][i])
    sum_N1_R1_C=0
    for j in range(len(disk_N1_R1_C)): #disk[j,0]:
        row=disk_N1_R1_C[j,0]
        col=disk_N1_R1_C[j,1]
        sum_N1_R1_C = sum_N1_R1_C + N1_R1_C['sky'][int(row),int(col),int(i)]
    sflux_N1_R1_C.append(sum_N1_R1_C)

In [11]:
vlambda_N1_R1_tot = []
vflux_N1_R1_tot = []
for i in range(len(N1_R1_tot['wavenumber'])):
    vlambda_N1_R1_tot.append(N1_R1_tot['wavenumber'][i])
    sum_N1_R1_tot=0
    for j in range(len(disk_N1_R1_tot)): #disk[j,0]:
        row=disk_N1_R1_tot[j,0]
        col=disk_N1_R1_tot[j,1]
        sum_N1_R1_tot = sum_N1_R1_tot + N1_R1_tot['spec_cube'][int(row),int(col),int(i)]
    vflux_N1_R1_tot.append(sum_N1_R1_tot)

slambda_N1_R1_tot = []
sflux_N1_R1_tot = []

for i in range(len(N1_R1_tot['wavenumber'])):
    slambda_N1_R1_tot.append(N1_R1_tot['wavenumber'][i])
    sum_N1_R1_tot=0
    for j in range(len(disk_N1_R1_tot)): #disk[j,0]:
        row=disk_N1_R1_tot[j,0]
        col=disk_N1_R1_tot[j,1]
        sum_N1_R1_tot = sum_N1_R1_tot + N1_R1_tot['sky'][int(row),int(col),int(i)]
    sflux_N1_R1_tot.append(sum_N1_R1_tot)

In [12]:
vlambda_N2_R1_A = []
vflux_N2_R1_A = []
for i in range(len(N2_R1_A['wavenumber'])):
    vlambda_N2_R1_A.append(N2_R1_A['wavenumber'][i])
    sum_N2_R1_A=0
    for j in range(len(disk_N2_R1_A)): #disk[j,0]:
        row=disk_N2_R1_A[j,0]
        col=disk_N2_R1_A[j,1]
        sum_N2_R1_A = sum_N2_R1_A + N2_R1_A['spec_cube'][int(row),int(col),int(i)]
    vflux_N2_R1_A.append(sum_N2_R1_A)

slambda_N2_R1_A = []
sflux_N2_R1_A = []

for i in range(len(N2_R1_A['wavenumber'])):
    slambda_N2_R1_A.append(N2_R1_A['wavenumber'][i])
    sum_N2_R1_A=0
    for j in range(len(disk_N2_R1_A)): #disk[j,0]:
        row=disk_N2_R1_A[j,0]
        col=disk_N2_R1_A[j,1]
        sum_N2_R1_A = sum_N2_R1_A + N2_R1_A['sky'][int(row),int(col),int(i)]
    sflux_N2_R1_A.append(sum_N2_R1_A)

In [13]:
vlambda_N2_R1_B = []
vflux_N2_R1_B = []
for i in range(len(N2_R1_B['wavenumber'])):
    vlambda_N2_R1_B.append(N2_R1_B['wavenumber'][i])
    sum_N2_R1_B=0
    for j in range(len(disk_N2_R1_B)): #disk[j,0]:
        row=disk_N2_R1_B[j,0]
        col=disk_N2_R1_B[j,1]
        sum_N2_R1_B = sum_N2_R1_B + N2_R1_B['spec_cube'][int(row),int(col),int(i)]
    vflux_N2_R1_B.append(sum_N2_R1_B)

slambda_N2_R1_B = []
sflux_N2_R1_B = []

for i in range(len(N2_R1_B['wavenumber'])):
    slambda_N2_R1_B.append(N2_R1_B['wavenumber'][i])
    sum_N2_R1_B=0
    for j in range(len(disk_N2_R1_B)): #disk[j,0]:
        row=disk_N2_R1_B[j,0]
        col=disk_N2_R1_B[j,1]
        sum_N2_R1_B = sum_N2_R1_B + N2_R1_B['sky'][int(row),int(col),int(i)]
    sflux_N2_R1_B.append(sum_N2_R1_B)

In [14]:
vlambda_N2_R1_C = []
vflux_N2_R1_C = []
for i in range(len(N2_R1_C['wavenumber'])):
    vlambda_N2_R1_C.append(N2_R1_C['wavenumber'][i])
    sum_N2_R1_C=0
    for j in range(len(disk_N2_R1_C)): #disk[j,0]:
        row=disk_N2_R1_C[j,0]
        col=disk_N2_R1_C[j,1]
        sum_N2_R1_C = sum_N2_R1_C + N2_R1_C['spec_cube'][int(row),int(col),int(i)]
    vflux_N2_R1_C.append(sum_N2_R1_C)

slambda_N2_R1_C = []
sflux_N2_R1_C = []

for i in range(len(N2_R1_C['wavenumber'])):
    slambda_N2_R1_C.append(N2_R1_A['wavenumber'][i])
    sum_N2_R1_C=0
    for j in range(len(disk_N2_R1_C)): #disk[j,0]:
        row=disk_N2_R1_C[j,0]
        col=disk_N2_R1_C[j,1]
        sum_N2_R1_C = sum_N2_R1_C + N2_R1_C['sky'][int(row),int(col),int(i)]
    sflux_N2_R1_C.append(sum_N2_R1_C)

In [15]:
vlambda_N2_R1_tot = []
vflux_N2_R1_tot = []
for i in range(len(N2_R1_tot['wavenumber'])):
    vlambda_N2_R1_tot.append(N2_R1_tot['wavenumber'][i])
    sum_N2_R1_tot=0
    for j in range(len(disk_N2_R1_tot)): #disk[j,0]:
        row=disk_N2_R1_tot[j,0]
        col=disk_N2_R1_tot[j,1]
        sum_N2_R1_tot = sum_N2_R1_tot + N2_R1_tot['spec_cube'][int(row),int(col),int(i)]
    vflux_N2_R1_tot.append(sum_N2_R1_tot)

slambda_N2_R1_tot = []
sflux_N2_R1_tot = []

for i in range(len(N2_R1_tot['wavenumber'])):
    slambda_N2_R1_tot.append(N2_R1_tot['wavenumber'][i])
    sum_N2_R1_tot=0
    for j in range(len(disk_N2_R1_tot)): #disk[j,0]:
        row=disk_N2_R1_tot[j,0]
        col=disk_N2_R1_tot[j,1]
        sum_N2_R1_tot = sum_N2_R1_tot + N2_R1_tot['sky'][int(row),int(col),int(i)]
    sflux_N2_R1_tot.append(sum_N2_R1_tot)

### Region Two

In [16]:
disk_N1_R2_A =np.argwhere(N1_R2_A.airmass>0)
disk_N1_R2_B =np.argwhere(N1_R2_B.airmass>0)
disk_N1_R2_tot =np.argwhere(N1_R2_tot.airmass>0)

disk_N2_R2_A =np.argwhere(N2_R2_A.airmass>0)
disk_N2_R2_B =np.argwhere(N2_R2_B.airmass>0)
disk_N2_R2_C =np.argwhere(N2_R2_C.airmass>0)
disk_N2_R2_tot =np.argwhere(N2_R2_tot.airmass>0)

In [17]:
vlambda_N1_R2_A = []
vflux_N1_R2_A = []
for i in range(len(N1_R2_A['wavenumber'])):
    vlambda_N1_R2_A.append(N1_R2_A['wavenumber'][i])
    sum_N1_R2_A=0
    for j in range(len(disk_N1_R2_A)): #disk[j,0]:
        row=disk_N1_R2_A[j,0]
        col=disk_N1_R2_A[j,1]
        sum_N1_R2_A = sum_N1_R2_A + N1_R2_A['spec_cube'][int(row),int(col),int(i)]
    vflux_N1_R2_A.append(sum_N1_R2_A)

slambda_N1_R2_A = []
sflux_N1_R2_A = []

for i in range(len(N1_R2_A['wavenumber'])):
    slambda_N1_R2_A.append(N1_R2_A['wavenumber'][i])
    sum_N1_R2_A=0
    for j in range(len(disk_N1_R2_A)): #disk[j,0]:
        row=disk_N1_R2_A[j,0]
        col=disk_N1_R2_A[j,1]
        sum_N1_R2_A = sum_N1_R2_A + N1_R2_A['sky'][int(row),int(col),int(i)]
    sflux_N1_R2_A.append(sum_N1_R2_A)

In [18]:
vlambda_N1_R2_B = []
vflux_N1_R2_B = []
for i in range(len(N1_R2_B['wavenumber'])):
    vlambda_N1_R2_B.append(N1_R2_B['wavenumber'][i])
    sum_N1_R2_B=0
    for j in range(len(disk_N1_R2_B)): #disk[j,0]:
        row=disk_N1_R2_B[j,0]
        col=disk_N1_R2_B[j,1]
        sum_N1_R2_B = sum_N1_R2_B + N1_R2_B['spec_cube'][int(row),int(col),int(i)]
    vflux_N1_R2_B.append(sum_N1_R2_B)

slambda_N1_R2_B = []
sflux_N1_R2_B = []

for i in range(len(N1_R2_B['wavenumber'])):
    slambda_N1_R2_B.append(N1_R2_B['wavenumber'][i])
    sum_N1_R2_B=0
    for j in range(len(disk_N1_R2_B)): #disk[j,0]:
        row=disk_N1_R2_B[j,0]
        col=disk_N1_R2_B[j,1]
        sum_N1_R2_B = sum_N1_R2_B + N1_R2_B['sky'][int(row),int(col),int(i)]
    sflux_N1_R2_B.append(sum_N1_R2_B)

In [19]:
vlambda_N1_R2_tot = []
vflux_N1_R2_tot = []
for i in range(len(N1_R2_tot['wavenumber'])):
    vlambda_N1_R2_tot.append(N1_R2_tot['wavenumber'][i])
    sum_N1_R2_tot=0
    for j in range(len(disk_N1_R2_tot)): #disk[j,0]:
        row=disk_N1_R2_tot[j,0]
        col=disk_N1_R2_tot[j,1]
        sum_N1_R2_tot = sum_N1_R2_tot + N1_R2_tot['spec_cube'][int(row),int(col),int(i)]
    vflux_N1_R2_tot.append(sum_N1_R2_tot)

slambda_N1_R2_tot = []
sflux_N1_R2_tot = []

for i in range(len(N1_R2_tot['wavenumber'])):
    slambda_N1_R2_tot.append(N1_R2_tot['wavenumber'][i])
    sum_N1_R2_tot=0
    for j in range(len(disk_N1_R2_tot)): #disk[j,0]:
        row=disk_N1_R2_tot[j,0]
        col=disk_N1_R2_tot[j,1]
        sum_N1_R2_tot = sum_N1_R2_tot + N1_R2_tot['sky'][int(row),int(col),int(i)]
    sflux_N1_R2_tot.append(sum_N1_R2_tot)

In [20]:
vlambda_N2_R2_A = []
vflux_N2_R2_A = []
for i in range(len(N2_R2_A['wavenumber'])):
    vlambda_N2_R2_A.append(N2_R2_A['wavenumber'][i])
    sum_N2_R2_A=0
    for j in range(len(disk_N2_R2_A)): #disk[j,0]:
        row=disk_N2_R2_A[j,0]
        col=disk_N2_R2_A[j,1]
        sum_N2_R2_A = sum_N2_R2_A + N2_R2_A['spec_cube'][int(row),int(col),int(i)]
    vflux_N2_R2_A.append(sum_N2_R2_A)

slambda_N2_R2_A = []
sflux_N2_R2_A = []

for i in range(len(N2_R2_A['wavenumber'])):
    slambda_N2_R2_A.append(N2_R2_A['wavenumber'][i])
    sum_N2_R2_A=0
    for j in range(len(disk_N2_R2_A)): #disk[j,0]:
        row=disk_N2_R2_A[j,0]
        col=disk_N2_R2_A[j,1]
        sum_N2_R2_A = sum_N2_R2_A + N2_R2_A['sky'][int(row),int(col),int(i)]
    sflux_N2_R2_A.append(sum_N2_R2_A)

In [21]:
vlambda_N2_R2_B = []
vflux_N2_R2_B = []
for i in range(len(N2_R2_B['wavenumber'])):
    vlambda_N2_R2_B.append(N2_R2_B['wavenumber'][i])
    sum_N2_R2_B=0
    for j in range(len(disk_N2_R2_B)): #disk[j,0]:
        row=disk_N2_R2_B[j,0]
        col=disk_N2_R2_B[j,1]
        sum_N2_R2_B = sum_N2_R2_B + N2_R2_B['spec_cube'][int(row),int(col),int(i)]
    vflux_N2_R2_B.append(sum_N2_R2_B)

slambda_N2_R2_B = []
sflux_N2_R2_B = []

for i in range(len(N2_R2_B['wavenumber'])):
    slambda_N2_R2_B.append(N2_R2_B['wavenumber'][i])
    sum_N2_R2_B=0
    for j in range(len(disk_N2_R2_B)): #disk[j,0]:
        row=disk_N2_R2_B[j,0]
        col=disk_N2_R2_B[j,1]
        sum_N2_R2_B = sum_N2_R2_B + N2_R2_B['sky'][int(row),int(col),int(i)]
    sflux_N2_R2_B.append(sum_N2_R2_B)

In [22]:
vlambda_N2_R2_C = []
vflux_N2_R2_C = []
for i in range(len(N2_R2_C['wavenumber'])):
    vlambda_N2_R2_C.append(N2_R2_C['wavenumber'][i])
    sum_N2_R2_C=0
    for j in range(len(disk_N2_R2_C)): #disk[j,0]:
        row=disk_N2_R2_C[j,0]
        col=disk_N2_R2_C[j,1]
        sum_N2_R2_C = sum_N2_R2_C + N2_R2_C['spec_cube'][int(row),int(col),int(i)]
    vflux_N2_R2_C.append(sum_N2_R2_C)

slambda_N2_R2_C = []
sflux_N2_R2_C = []

for i in range(len(N2_R2_C['wavenumber'])):
    slambda_N2_R2_C.append(N2_R2_A['wavenumber'][i])
    sum_N2_R2_C=0
    for j in range(len(disk_N2_R2_C)): #disk[j,0]:
        row=disk_N2_R2_C[j,0]
        col=disk_N2_R2_C[j,1]
        sum_N2_R2_C = sum_N2_R2_C + N2_R2_C['sky'][int(row),int(col),int(i)]
    sflux_N2_R2_C.append(sum_N2_R2_C)

In [23]:
vlambda_N2_R2_tot = []
vflux_N2_R2_tot = []
for i in range(len(N2_R2_tot['wavenumber'])):
    vlambda_N2_R2_tot.append(N2_R2_tot['wavenumber'][i])
    sum_N2_R2_tot=0
    for j in range(len(disk_N2_R2_tot)): #disk[j,0]:
        row=disk_N2_R2_tot[j,0]
        col=disk_N2_R2_tot[j,1]
        sum_N2_R2_tot = sum_N2_R2_tot + N2_R2_tot['spec_cube'][int(row),int(col),int(i)]
    vflux_N2_R2_tot.append(sum_N2_R2_tot)

slambda_N2_R2_tot = []
sflux_N2_R2_tot = []

for i in range(len(N2_R2_tot['wavenumber'])):
    slambda_N2_R2_tot.append(N2_R2_tot['wavenumber'][i])
    sum_N2_R2_tot=0
    for j in range(len(disk_N2_R2_tot)): #disk[j,0]:
        row=disk_N2_R2_tot[j,0]
        col=disk_N2_R2_tot[j,1]
        sum_N2_R2_tot = sum_N2_R2_tot + N2_R2_tot['sky'][int(row),int(col),int(i)]
    sflux_N2_R2_tot.append(sum_N2_R2_tot)

# Plotting

## Ranges

In [24]:
vrange_N1_R1_A = (np.max(vflux_N1_R1_A))-(np.min(vflux_N1_R1_A))
print(vrange_N1_R1_A)
print(np.max(vflux_N1_R1_A))
print(np.min(vflux_N1_R2_tot))
srange_N1_R1_A = (np.max(sflux_N1_R1_A))-(np.min(sflux_N1_R1_A))

vrange_N1_R1_B = (np.max(vflux_N1_R1_B))-(np.min(vflux_N1_R1_B))
srange_N1_R1_B = (np.max(sflux_N1_R1_B))-(np.min(sflux_N1_R1_B))

vrange_N1_R1_C = (np.max(vflux_N1_R1_C))-(np.min(vflux_N1_R1_C))
srange_N1_R1_C = (np.max(sflux_N1_R1_C))-(np.min(sflux_N1_R1_C))

vrange_N1_R1_tot = (np.max(vflux_N1_R1_tot))-(np.min(vflux_N1_R1_tot))
srange_N1_R1_tot = (np.max(sflux_N1_R1_tot))-(np.min(sflux_N1_R1_tot))

5098.155027389526
5098.155027389526
0.0


In [25]:
vrange_N2_R1_A = (np.max(vflux_N1_R1_A))-(np.min(vflux_N1_R1_A))
srange_N2_R1_A = (np.max(sflux_N1_R1_A))-(np.min(sflux_N1_R1_A))

vrange_N2_R1_B = (np.max(vflux_N1_R1_B))-(np.min(vflux_N1_R1_B))
srange_N2_R1_B = (np.max(sflux_N1_R1_B))-(np.min(sflux_N1_R1_B))

vrange_N2_R1_C = (np.max(vflux_N1_R1_C))-(np.min(vflux_N1_R1_C))
srange_N2_R1_C = (np.max(sflux_N1_R1_C))-(np.min(sflux_N1_R1_C))

vrange_N2_R1_tot = (np.max(vflux_N1_R1_tot))-(np.min(vflux_N1_R1_tot))
srange_N2_R1_tot = (np.max(sflux_N1_R1_tot))-(np.min(sflux_N1_R1_tot))

In [26]:
vrange_N1_R2_A = (np.max(vflux_N1_R1_A))-(np.min(vflux_N1_R1_A))
srange_N1_R2_A = (np.max(sflux_N1_R1_A))-(np.min(sflux_N1_R1_A))

vrange_N1_R2_B = (np.max(vflux_N1_R1_B))-(np.min(vflux_N1_R1_B))
srange_N1_R2_B = (np.max(sflux_N1_R1_B))-(np.min(sflux_N1_R1_B))

vrange_N1_R2_tot = (np.max(vflux_N1_R1_tot))-(np.min(vflux_N1_R1_tot))
srange_N1_R2_tot = (np.max(sflux_N1_R1_tot))-(np.min(sflux_N1_R1_tot))

In [27]:
vrange_N2_R2_A = (np.max(vflux_N1_R1_A))-(np.min(vflux_N1_R1_A))
srange_N2_R2_A = (np.max(sflux_N1_R1_A))-(np.min(sflux_N1_R1_A))

vrange_N2_R2_B = (np.max(vflux_N1_R1_B))-(np.min(vflux_N1_R1_B))
srange_N2_R2_B = (np.max(sflux_N1_R1_B))-(np.min(sflux_N1_R1_B))

vrange_N2_R2_C = (np.max(vflux_N1_R1_C))-(np.min(vflux_N1_R1_C))
srange_N2_R2_C = (np.max(sflux_N1_R1_C))-(np.min(sflux_N1_R1_C))

vrange_N2_R2_tot = (np.max(vflux_N1_R1_tot))-(np.min(vflux_N1_R1_tot))
srange_N2_R2_tot = (np.max(sflux_N1_R1_tot))-(np.min(sflux_N1_R1_tot))

## Night One Region One

In [28]:
TOOLTIPS = [ ("x","$x{‘00.00000000’}"),("y","$y{‘00.000’}"), ("strength", "$name")]

q = figure(title="Night One Region One",sizing_mode="stretch_width",
    height=550,tooltips=TOOLTIPS,
    x_axis_label="Wavenumber (cm^(-1))",
    y_axis_label="Intensity")


#q.line(vlambda_N1_R1_A,-.01+(((vflux_N1_R1_A-(np.min(vflux_N1_R1_A)))*(1+.01))/ vrange_N1_R1_A),legend_label='Venus N1 R1 A', color='red')
#q.line(vlambda_N1_R1_B,-.01+(((vflux_N1_R1_B-(np.min(vflux_N1_R1_B)))*(1+.01))/ vrange_N1_R1_B),legend_label='Venus N1 R1 B', color='red')
#q.line(vlambda_N1_R1_C,-.01+(((vflux_N1_R1_C-(np.min(vflux_N1_R1_C)))*(1+.01))/ vrange_N1_R1_C),legend_label='Venus N1 R1 C', color='red')
q.line(vlambda_N1_R1_tot,-.01+(((vflux_N1_R1_tot-(np.min(vflux_N1_R1_tot)))*(1+.01))/ vrange_N1_R1_tot),legend_label='Venus N1 R1 Total', color='red')

#q.line(slambda_N1_R1_A,-.01+(((sflux_N1_R1_A-(np.min(sflux_N1_R1_A)))*(1+.01))/ srange_N1_R1_A),legend_label='Sky N1 R1 A', color='blue')
#q.line(slambda_N1_R1_B,-.01+(((sflux_N1_R1_B-(np.min(sflux_N1_R1_B)))*(1+.01))/ srange_N1_R1_B),legend_label='Sky N1 R1 B', color='blue')
#q.line(slambda_N1_R1_C,-.01+(((sflux_N1_R1_C-(np.min(sflux_N1_R1_C)))*(1+.01))/ srange_N1_R1_C),legend_label='Sky N1 R1 C', color='blue')
q.line(slambda_N1_R1_tot,-.01+(((sflux_N1_R1_tot-(np.min(sflux_N1_R1_tot)))*(1+.01))/ srange_N1_R1_tot),legend_label='Sky N1 R1 Total', color='blue')


q.legend.location = "bottom_right"

show(q)

## Night Two Region One

In [29]:
TOOLTIPS = [ ("x","$x{‘00.00000000’}"),("y","$y{‘00.000’}"), ("strength", "$name")]

q = figure(title="Night Two Region One",sizing_mode="stretch_width",
    height=550,tooltips=TOOLTIPS,
    x_axis_label="Wavenumber (cm^(-1))",
    y_axis_label="Intensity")


#q.line(vlambda_N2_R1_A,-.01+(((vflux_N2_R1_A-(np.min(vflux_N2_R1_A)))*(1+.01))/ vrange_N2_R1_A),legend_label='Venus N2 R1 A', color='red')
#q.line(vlambda_N2_R1_B,-.01+(((vflux_N2_R1_B-(np.min(vflux_N2_R1_B)))*(1+.01))/ vrange_N2_R1_B),legend_label='Venus N2 R1 B', color='red')
#q.line(vlambda_N2_R1_C,-.01+(((vflux_N2_R1_C-(np.min(vflux_N2_R1_C)))*(1+.01))/ vrange_N2_R1_C),legend_label='Venus N2 R1 C', color='red')
q.line(vlambda_N2_R1_tot,-.01+(((vflux_N2_R1_tot-(np.min(vflux_N2_R1_tot)))*(1+.01))/ vrange_N2_R1_tot),legend_label='Venus N2 R1 Total', color='red')

#q.line(slambda_N2_R1_A,-.01+(((sflux_N2_R1_A-(np.min(sflux_N2_R1_A)))*(1+.01))/ srange_N2_R1_A),legend_label='Sky N2 R1 A', color='blue')
#q.line(slambda_N2_R1_B,-.01+(((sflux_N2_R1_B-(np.min(sflux_N2_R1_B)))*(1+.01))/ srange_N2_R1_B),legend_label='Sky N2 R1 B', color='blue')
#q.line(slambda_N2_R1_C,-.01+(((sflux_N2_R1_C-(np.min(sflux_N2_R1_C)))*(1+.01))/ srange_N2_R1_C),legend_label='Sky N2 R1 C', color='blue')
q.line(slambda_N2_R1_tot,-.01+(((sflux_N2_R1_tot-(np.min(sflux_N2_R1_tot)))*(1+.01))/ srange_N2_R1_tot),legend_label='Sky N2 R1 Total', color='blue')


q.legend.location = "bottom_right"

show(q)

## Night One Region Two

In [30]:
TOOLTIPS = [ ("x","$x{‘00.00000000’}"),("y","$y{‘00.000’}"), ("strength", "$name")]

q = figure(title="Night One Region Two",sizing_mode="stretch_width",
    height=550,tooltips=TOOLTIPS,
    x_axis_label="Wavenumber (cm^(-1))",
    y_axis_label="Intensity")


#q.line(vlambda_N1_R2_A,-.01+(((vflux_N1_R2_A-(np.min(vflux_N1_R2_A)))*(1+.01))/ vrange_N1_R2_A),legend_label='Venus N1 R2 A', color='red')
#q.line(vlambda_N1_R2_B,-.01+(((vflux_N1_R2_B-(np.min(vflux_N1_R2_B)))*(1+.01))/ vrange_N1_R2_B),legend_label='Venus N1 R2 B', color='red')
q.line(vlambda_N1_R2_tot,-.01+(((vflux_N1_R2_tot-(np.min(vflux_N1_R2_tot)))*(1+.01))/ vrange_N1_R2_tot),legend_label='Venus N1 R2 Total', color='red')

#q.line(slambda_N1_R2_A,-.01+(((sflux_N1_R2_A-(np.min(sflux_N1_R2_A)))*(1+.01))/ srange_N1_R2_A),legend_label='Sky N1 R2 A', color='blue')
#q.line(slambda_N1_R2_B,-.01+(((sflux_N1_R2_B-(np.min(sflux_N1_R2_B)))*(1+.01))/ srange_N1_R2_B),legend_label='Sky N1 R2 B', color='blue')
q.line(slambda_N1_R2_tot,-.01+(((sflux_N1_R2_tot-(np.min(sflux_N1_R2_tot)))*(1+.01))/ srange_N1_R2_tot),legend_label='Sky N1 R2 Total', color='blue')


q.legend.location = "bottom_right"

show(q)

## Night Two Region Two

In [31]:
TOOLTIPS = [ ("x","$x{‘00.00000000’}"),("y","$y{‘00.000’}"), ("strength", "$name")]

q = figure(title="Night Two Region Two",sizing_mode="stretch_width",
    height=550,tooltips=TOOLTIPS,
    x_axis_label="Wavenumber (cm^(-1))",
    y_axis_label="Intensity")


#q.line(vlambda_N2_R2_A,-.01+(((vflux_N2_R2_A-(np.min(vflux_N2_R2_A)))*(1+.01))/ vrange_N2_R2_A),legend_label='Venus N2 R2 A', color='red')
#q.line(vlambda_N2_R2_B,-.01+(((vflux_N2_R2_B-(np.min(vflux_N2_R2_B)))*(1+.01))/ vrange_N2_R2_B),legend_label='Venus N2 R2 B', color='red')
#q.line(vlambda_N2_R2_C,-.01+(((vflux_N2_R2_C-(np.min(vflux_N2_R2_C)))*(1+.01))/ vrange_N2_R2_C),legend_label='Venus N2 R2 C', color='red')
q.line(vlambda_N2_R2_tot,-.01+(((vflux_N2_R2_tot-(np.min(vflux_N2_R2_tot)))*(1+.01))/ vrange_N2_R2_tot),legend_label='Venus N2 R2 Total', color='red')

#q.line(slambda_N2_R2_A,-.01+(((sflux_N2_R2_A-(np.min(sflux_N2_R2_A)))*(1+.01))/ srange_N2_R2_A),legend_label='Sky N2 R2 A', color='blue')
#q.line(slambda_N2_R2_B,-.01+(((sflux_N2_R2_B-(np.min(sflux_N2_R2_B)))*(1+.01))/ srange_N2_R2_B),legend_label='Sky N2 R2 B', color='blue')
#q.line(slambda_N2_R2_C,-.01+(((sflux_N2_R2_C-(np.min(sflux_N2_R2_C)))*(1+.01))/ srange_N2_R2_C),legend_label='Sky N2 R2 C', color='blue')
q.line(slambda_N2_R2_tot,-.01+(((sflux_N2_R2_tot-(np.min(sflux_N2_R2_tot)))*(1+.01))/ srange_N2_R2_tot),legend_label='Sky N2 R2 Total', color='blue')


q.legend.location = "bottom_right"

show(q)

# PSG Spectra

In [32]:
#setting variables, one is the PSG wavenumber, one is the composite spectrum, and the rest are the individual spectra that make up PSG's composite spectrum
R1_rev = []
R1_tot_rev = []
R1_H2O_rev = []
R1_CO2_rev = []
R1_O3_rev = []
R1_N2O_rev = []
R1_CO_rev = []
R1_CH4_rev = []
R1_O2_rev = []
R1_N2_rev = []
R1_Rayleigh_rev = []
R1_CIA_rev = []

#defining each variable with a row contained in the PSG .csv file
with open('/Users/physicsstudent2/desktop/venus_research/BSRI_2024/PSG/R1_PSG.csv') as r1_psg_file:  
    plots = csv.reader(r1_psg_file, delimiter = ',') 
      
    for row in plots: 
        R1_rev.append(float(row[0]))
        R1_tot_rev.append(float(row[1]))
        R1_H2O_rev.append(float(row[2]))
        R1_CO2_rev.append(float(row[3]))
        R1_O3_rev.append(float(row[4]))
        R1_N2O_rev.append(float(row[5]))
        R1_CO_rev.append(float(row[6]))
        R1_CH4_rev.append(float(row[7]))
        R1_O2_rev.append(float(row[8]))
        R1_N2_rev.append(float(row[9]))
        R1_Rayleigh_rev.append(float(row[10]))
        R1_CIA_rev.append(float(row[11]))

In [33]:
TOOLTIPS = [ ("x","$x{‘00.00000000’}"),("y","$y{‘00.000’}"), ("strength", "$name")]
q = figure(title="Region One PSG",sizing_mode="stretch_width",
    height=500,tooltips=TOOLTIPS,
    x_axis_label="Wavenumber (cm^(-1))",
    y_axis_label="Intensity")

q.line(R1_rev, R1_tot_rev, legend_label='Earth', color='thistle') #consists of H2O, O3, N2O, and CH4
q.line(R1_rev, R1_H2O_rev,legend_label='H2O', color='navy') #three features present
q.line(R1_rev, R1_CO2_rev,legend_label='CO2', color='navajowhite')
#q.line(R1_rev, R1_O3_rev,legend_label='O3', color='darkseagreen') #primariliy present
q.line(R1_rev, R1_N2O_rev,legend_label='N2O', color='lavender') #largely present
q.line(R1_rev, R1_CO_rev,legend_label='CO', color='khaki') 
q.line(R1_rev, R1_CH4_rev,legend_label='CH4', color='lightcoral') #present 
#q.line(R1_rev, R1_O2_rev,legend_label='O2', color='lightskyblue')
q.line(R1_rev, R1_N2_rev,legend_label='N2', color='mediumseagreen')
q.line(R1_rev, R1_Rayleigh_rev,legend_label='Rayleigh', color='rosybrown')
q.line(R1_rev, R1_CIA_rev,legend_label='CIA', color='burlywood')


q.legend.location = "bottom_right"

show(q)

# Data Processing

## Variables

In [342]:
R1_lambda = list(reversed(R1_rev))
R1_lambdads = []
for number in R1_lambda:
    R1_lambdads.append(number+0.0315159369) #Plus the Doppler Shift

R1_H2O = list(reversed(R1_H2O_rev))
R1_O3 = list(reversed(R1_O3_rev))
R1_N2O = list(reversed(R1_N2O_rev))
R1_CH4 = list(reversed(R1_CH4_rev))

psg_step_size = ((max(R1_lambdads)-min(R1_lambdads))/len(R1_lambdads))
print(psg_step_size)
step_N1_R1_tot = ((max(vlambda_N1_R1_tot)-min(vlambda_N1_R1_tot))/len(vlambda_N1_R1_tot))
print(step_N1_R1_tot)

0.0022354835680750956
0.0037142318573480885


## Night One Region One Total

In [343]:
vnorm_N1_R1_tot = -.01+(((vflux_N1_R1_tot-(np.min(vflux_N1_R1_tot)))*(1+.01))/ vrange_N1_R1_tot)
snorm_N1_R1_tot = -.01+(((sflux_N1_R1_tot-(np.min(sflux_N1_R1_tot)))*(1+.01))/ srange_N1_R1_tot)

In [344]:
print(min(R1_H2O))
print(R1_H2O.index(0.057861))
print(R1_lambda[3683])
print(min(vnorm_N1_R1_tot[1900:2000]))
print(np.where(vnorm_N1_R1_tot == 0.30934588450901235))
print(vlambda_N1_R1_tot[1967])

#print(vlambda_N1_R1_tot[1900:2000]) 
print(len(vnorm_N1_R1_tot))
print(len(vflux_N2_R2_tot))

H2Oexp_N1_R1_tot = np.log(min(vnorm_N1_R1_tot[1900:2000]))/np.log(min(R1_H2O)) + 0.1
print(H2Oexp_N1_R1_tot)
print(vlambda_N1_R1_tot[1967]-R1_lambda[3683])

0.057861
3683
1121.229305
0.30934588450901235
(array([1967]),)
1121.2655949026803
2385
1732
0.5117241972977327
0.03628990268020971


In [345]:
print(min(R1_O3))
print(R1_O3.index(0.0513534))
print(R1_lambda[3475])

#print(vlambda_N1_R1_tot[1800:1900])
print(min(vnorm_N1_R1_tot[1800:1900]))
#i feel like this might be inaccurate if another molecule produces a minimum in the spectrum in the same range

#O3exp_N1_R1_tot = np.log(min(vnorm_N1_R1_tot[1800:1900]))/np.log(min(R1_O3)) + 0.4
O3exp_N1_R1_tot = 0.68
print(O3exp_N1_R1_tot)

0.0513534
3475
1120.762971
0.4810451686350986
0.68


In [346]:
print(min(R1_N2O))
print(R1_N2O.index(0.999004))
print(R1_lambda[4362])

#print(vlambda_N1_R1_tot[2050:2150])
print(min(vnorm_N1_R1_tot[2050:2150]))

#N2Oexp_N1_R1_tot = np.log(min(vnorm_N1_R1_tot[2050:2150]))/np.log(min(R1_N2O))
N2Oexp_N1_R1_tot = 0.1
print(N2Oexp_N1_R1_tot)

0.999004
4362
1122.752967
0.6561369092180316
0.1


In [347]:
print(min(R1_CH4))
print(R1_CH4.index(0.992888))
print(R1_lambda[4194])

#print(vlambda_N1_R1_tot[1950:2050])
print(min(vnorm_N1_R1_tot[1950:2050]))

#CH4exp_N1_R1_tot = np.log(min(vnorm_N1_R1_tot[2050:2150]))/np.log(min(R1_CH4))
CH4exp_N1_R1_tot = 0.1
print(CH4exp_N1_R1_tot)

0.992888
4194
1122.375786
0.30934588450901235
0.1


In [348]:
k_N1_R1_tot = 2 #Gaussian sigma of resolution broadening, in pixels
lambshift_N1_R1_tot = 0 #empirical wavenumber shift to improve residuals

In [349]:
#multiplying the spectra by their respective exponents
H2O_N1_R1_tot=[]
O3_N1_R1_tot=[]
N2O_N1_R1_tot=[]
CH4_N1_R1_tot=[]
for number in R1_H2O:
    H2O_N1_R1_tot.append(number**H2Oexp_N1_R1_tot)
for number in R1_O3:
    O3_N1_R1_tot.append(number**O3exp_N1_R1_tot)
for number in R1_N2O:
    N2O_N1_R1_tot.append(number**N2Oexp_N1_R1_tot)
for number in R1_CH4:
    CH4_N1_R1_tot.append(number**CH4exp_N1_R1_tot)

#multiplying the exponentiated spectra together to create a new composite spectrum
tot_N1_R1_tot = np.array(H2O_N1_R1_tot)*np.array(O3_N1_R1_tot)*np.array(N2O_N1_R1_tot)*np.array(CH4_N1_R1_tot)

#gaussian smoothing the spectrum
gaus_N1_R1_tot = gaussian_filter(tot_N1_R1_tot,k_N1_R1_tot)

#interpolating the spectrum over the wavenumber range
int_N1_R1_tot = scipy.interpolate.interp1d(R1_lambdads, gaus_N1_R1_tot)
intfunc_N1_R1_tot = int_N1_R1_tot(vlambda_N1_R1_tot)
tell_N1_R1_tot = shift(intfunc_N1_R1_tot, lambshift_N1_R1_tot)

#normalizing the model
normtell_N1_R1_tot  = tell_N1_R1_tot / np.median(tell_N1_R1_tot)

#dividing the spectrum by the model
vproc_N1_R1_tot = np.divide(vnorm_N1_R1_tot, normtell_N1_R1_tot)

In [350]:
#plotting the processed file
TOOLTIPS = [ ("x","$x{‘00.00000000’}"),("y","$y{‘00.000’}"), ("strength", "$name")]
q = figure(title="File One",sizing_mode="stretch_width",
    height=500,tooltips=TOOLTIPS)

q.line(vlambda_N1_R1_tot, normtell_N1_R1_tot -0.1, legend_label='corrected telluric', color='orange')
q.line(vlambda_N1_R1_tot, vproc_N1_R1_tot + 0.1, legend_label='venus processed', color='green')
q.line(vlambda_N1_R1_tot, vnorm_N1_R1_tot, legend_label='venus unprocessed', color='red')
#q.line(vlambda_N1_R1_tot, snorm_N1_R1_tot,legend_label='sky', color='blue')

#for place in range(0,len(line_locations_region_1)):
    #q.line([range1_array[place,0]+0.03,range1_array[place,0]+0.03], [1.06,0.60],color='black',line_width=1,name=str(range1_array[place,1]), legend_label='Strong Telluric Features That Cannot Be Fully Processed')
#for place in range(0,len(two_line_locations_region_1)):
    #q.line([two_range1_array[place,0]+0.03,two_range1_array[place,0]+0.03], [1.06,0.60],color='black',line_width=1,name=str(two_range1_array[place,1]))

q.legend.location = "bottom_right"

show(q)

## Night Two Region One Total

In [355]:
vnorm_N2_R1_tot = -.01+(((vflux_N2_R1_tot-(np.min(vflux_N2_R1_tot)))*(1+.01))/ vrange_N2_R1_tot)
snorm_N2_R1_tot = -.01+(((sflux_N2_R1_tot-(np.min(sflux_N2_R1_tot)))*(1+.01))/ srange_N2_R1_tot)

In [359]:
#interpolating the spectrum over the wavenumber range
int_N2_R1_tot = scipy.interpolate.interp1d(R1_lambdads, gaus_N1_R1_tot)
intfunc_N2_R1_tot = int_N2_R1_tot(vlambda_N2_R1_tot)
tell_N2_R1_tot = shift(intfunc_N2_R1_tot, lambshift_N2_R1_tot)

#normalizing the model
normtell_N2_R1_tot  = tell_N2_R1_tot / np.median(tell_N2_R1_tot)

#dividing the spectrum by the model
vproc_N2_R1_tot = np.divide(vnorm_N2_R1_tot, normtell_N2_R1_tot)

In [360]:
#plotting the processed file
TOOLTIPS = [ ("x","$x{‘00.00000000’}"),("y","$y{‘00.000’}"), ("strength", "$name")]
q = figure(title="File One",sizing_mode="stretch_width",
    height=500,tooltips=TOOLTIPS)

q.line(vlambda_N2_R1_tot, normtell_N2_R1_tot -0.1, legend_label='corrected telluric', color='orange')
q.line(vlambda_N2_R1_tot, vproc_N2_R1_tot + 0.1, legend_label='venus processed', color='green')
q.line(vlambda_N2_R1_tot, vnorm_N2_R1_tot, legend_label='venus unprocessed', color='red')
#q.line(vlambda_N2_R1_tot, snorm_N2_R1_tot,legend_label='sky', color='blue')

#for place in range(0,len(line_locations_region_1)):
    #q.line([range1_array[place,0]+0.03,range1_array[place,0]+0.03], [1.06,0.60],color='black',line_width=1,name=str(range1_array[place,1]), legend_label='Strong Telluric Features That Cannot Be Fully Processed')
#for place in range(0,len(two_line_locations_region_1)):
    #q.line([two_range1_array[place,0]+0.03,two_range1_array[place,0]+0.03], [1.06,0.60],color='black',line_width=1,name=str(two_range1_array[place,1]))

q.legend.location = "bottom_right"

show(q)